In [ ]:
import deepinv as dinv
import torch
import matplotlib.pyplot as plt
from deepinv.utils.demo import load_dataset
from torchvision import transforms
import tqdm
import numpy as np
import arviz 

In [ ]:
def plot_MC_corr(X_trace, ax, title="X"):
    """ 
    Plot the autocorrelation of the MC X_trace if shape (n_samples, 1), and plot the autocorrelation of the min, median and
    max variance pixels if shape (n_samples, n_pixels). Displays the effective sampling size in plot title.
    """
    if X_trace.shape[1] > 1:  # shape (n_samples, n_pixels)
        X_var = np.var(X_trace, axis=0)   
        # track pixels with max, min, med variances
        sort_ind = np.argsort(X_var)
        X_max_var = X_trace[:, sort_ind[-1]]
        X_min_var = X_trace[:, sort_ind[0]]
        X_med_var = X_trace[:, sort_ind[len(sort_ind) // 2]]
        ds = dict(Xmax=X_max_var, Xmin=X_min_var, Xmed=X_med_var)
        # corresponding effective sample sizes
        ess_max = arviz.ess(X_max_var.reshape(-1))
        ess_min = arviz.ess(X_min_var.reshape(-1))
        ess_med  = arviz.ess(X_med_var.reshape(-1))
        arviz.plot_autocorr(ds, var_names=['Xmin', 'Xmed', 'Xmax'], ax=ax)
        ax[0].set_title(title + " min: {:.1f}".format(ess_min))
        ax[1].set_title(title + " med: {:.1f}".format(ess_med))
        ax[2].set_title(title + " max: {:.1f}".format(ess_max))
    else:  # shape (n_samples,)
        ess = arviz.ess(X_trace.reshape(-1))
        ds = dict(X=X_trace.reshape(-1))
        arviz.plot_autocorr(ds, var_names=['X'], ax=ax)
        ax[0].set_title(title + ": {:.1f}".format(ess))

In [ ]:
class ParametrizedPrior:
    def __init__(self, param):
        self.param = torch.clone(param).to(device)

    def forward(self, x, *args, **kwargs):
        pass

    def __call__(self, x, *args, **kwargs):
        return self.forward(x, **kwargs)
        
    def grad(self, x, lam_reg=None):
        pass

    def grad_param(self, x):
        pass
        
    def update_param(self, new_param):
        self.param = torch.clone(new_param)

class Likelihood:
    def __init__(self):
        pass

    def __call__(self, x, y):
        return self.f(x,y)
    
    def f(self, x, y):
        pass

    def grad(self, x, y):
        pass

class CombinedPrior(ParametrizedPrior):
    def __init__(self, param, prior1, prior2):
        super().__init__(param)
        self.prior1 = prior1
        self.prior2 = prior2

    def grad(self, x, lam_reg):
        return self.param * self.prior1.grad(x, lam_reg) + (1 - self.param) * self.prior2.grad(x, lam_reg)

    def forward(self, x):
        return self.param * self.prior1(x) + (1 - self.param) * self.prior2(x)

    def grad_param(self, x):
        return  self.prior1.grad_param(x) - self.prior2.grad_param(x)


class SAPG:
    def __init__(self, y, g, f, gamma, gammap, lam_reg, X_init=None):
        """
        y: observations
        f: likelihood
        g: prior
        """
        self.y = y.clone().to(device)
        self.dimx = y.numel()
        self.g, self.f = g, f  # - log prior and likelihood
        self.eta = torch.log(g.param).detach().to(device)  # e^eta = theta
        self.gamma, self.gammap = gamma.to(device), gammap.to(device)
        self.lam_reg = lam_reg.to(device)
        if lam_reg is None:  # regularization parameter for prior
            self.lamp_reg = None
        else:
            self.lamp_reg = 0.98*lam_reg

        if X_init is None:
            self.X_prior, self.X_post = torch.randn_like(y).to(device), torch.randn_like(y).to(device)
        else: self.X_prior, self.X_post = torch.clamp(X_init.clone(), 0, 1), torch.clamp(X_init.clone(), 0, 1)

        self.X_prior_warm, self.X_post_warm = None, None

    def _ULA_it_post(self):
        with torch.no_grad():
            self.X_post = torch.clamp(self.X_post - self.gamma * self.f.grad(self.X_post, self.y) - 
                                      self.gamma * self.g.grad(self.X_post, self.lam_reg) + 
                                      torch.sqrt(2*self.gamma) * torch.randn_like(self.X_post), 0., 1.)

    def _ULA_it_prior(self):
            self.X_prior = torch.clamp(self.X_prior - self.gammap * self.g.grad(self.X_prior, self.lamp_reg) + 
                                       torch.sqrt(2*self.gammap) * torch.randn_like(self.X_prior), 0., 1.)
            
    def sample_post(self, nb_it, X_init=None):
        if X_init is not None:
            X_post = X_init.clone()
        else:
            X_post = self.X_post.clone()

        hist, post_hist = torch.zeros((nb_it,) + X_post.shape, device=device), torch.zeros(nb_it, device=device)
        for n in tqdm.tqdm(range(nb_it)):
            with torch.no_grad():
                X_post = (X_post - self.gamma * self.f.grad(X_post, self.y) - 
                          self.gamma * self.g.grad(X_post, self.lam_reg) + torch.sqrt(2*self.gamma) * torch.randn_like(X_post))
                hist[n] = X_post.detach().clone()
                post_hist[n] = self.post(X_post, self.y).detach()
        return hist, post_hist
            
    def warm_up_post(self, nb_steps, log_stats=False, thinning=10, burnin_ratio=0.8):
        if log_stats:
            it_burnin, count_stats = int(burnin_ratio*nb_steps), 0
            n_rem = nb_steps - it_burnin
            post_hist = torch.zeros(n_rem // thinning, device=device)
            X_post_trace = torch.zeros([n_rem // thinning, self.dimx]).to(device)
        for n in tqdm.tqdm(range(nb_steps)):
            with torch.no_grad():
                self._ULA_it_post()
                
                if log_stats and n % thinning == 0 and n >= it_burnin:
                    X_post_trace[count_stats] =  torch.flatten(self.X_post).detach()
                    post_hist[count_stats] = self.post(self.X_post, y).detach()
                    count_stats += 1
                
        self.X_post_warm = self.X_post.clone()
        if log_stats:
            return X_post_trace.cpu(), post_hist[:count_stats].cpu()
        else:
            return

    def warm_up_prior(self, nb_steps, log_stats=False, thinning=10, burnin_ratio=0.8):
        if log_stats:
            it_burnin, count_stats = int(burnin_ratio*nb_steps), 0
            n_rem = nb_steps - it_burnin
            prior_hist = torch.zeros(n_rem // thinning, device=device)
            X_prior_trace = torch.zeros([n_rem // thinning, self.dimx]).to(device)

        for n in tqdm.tqdm(range(nb_steps)):
            with torch.no_grad():
                self._ULA_it_prior()
                
                if log_stats and n % thinning == 0 and n >= it_burnin:
                    X_prior_trace[count_stats] = torch.flatten(self.X_prior).detach()
                    prior_hist[count_stats] = - self.g(self.X_prior).detach()
                    count_stats += 1
                
        self.X_prior_warm = self.X_prior.clone()
        if log_stats:
            return X_prior_trace.cpu(), prior_hist[:count_stats].cpu()
        else:
            return

    def post(self, x, y):  # log posterior 
        return -self.f(x,y) - self.g(x)
        
    def update_param(self, new_param):
        self.eta = torch.log(new_param)
        self.g.update_param(new_param)
        
    def run(self, delta, nb_steps, bounds, init_param=None, thinning_global=1,
            burnin_ratio=0.8, thinning_post=10, thinning_prior=10, tol=1e-4, alpha=None,
            reuse_post=False):
        """
        delta: fun that updates gradient stepsize
        """
        if ((self.X_prior_warm is None) and (alpha is None)) or (self.X_post_warm is None):
            print("Warm up MC first")
            return 
        else:
            self.X_posterior = self.X_post_warm.clone()
            if alpha is None:
                self.X_prior = self.X_prior_warm.clone()

        if init_param is not None:
            self.update_param(init_param)
        else:
            self.update_param(self.g.param)  # to ensure eta is up to date
        log_bounds = torch.log(torch.tensor(bounds)).to(device)
        
        g = self.g
        nit_param = 0  # num of param updates - 1
        nit_mean = 0  # num of mean estimator updates - 1

        prior_hist = torch.zeros(nb_steps // thinning_global, device=device)
        post_hist = torch.zeros(nb_steps // thinning_global, device=device)
        mean_hist =  torch.zeros(nb_steps // thinning_global, device=device)
        param_hist = torch.zeros(nb_steps // thinning_global, device=device)
        
        it_burnin = int(burnin_ratio*nb_steps)  # first iteration for updating mean
        
        trange = tqdm.tqdm(range(1, nb_steps + 1))
        for n in trange:
            trange.set_description("theta={:.4f}, eta={:.4f}".format(g.param.item(), self.eta.item()))

            with torch.no_grad():

                for _ in range(thinning_post):
                    self._ULA_it_post()
                if alpha is None:
                    if reuse_post:
                        self.X_prior = self.X_post.clone()
                    for _ in range(thinning_prior):
                        self._ULA_it_prior()
                
                if n % thinning_global == 0:  # update parameters
                    step = delta(n)
                    grad_param_post = g.grad_param(self.X_post)
                    if alpha is None:
                        grad_param_prior = g.grad_param(self.X_prior)
                    else:
                        grad_param_prior = self.dimx / alpha / self.g.param
                    param_hist[nit_param] = g.param.detach().clone()
                    prior_hist[nit_param] = - grad_param_prior
                    post_hist[nit_param] = - grad_param_post
                    
                    # gradient step in logarithmic scale
                    self.eta = torch.clamp(self.eta +   # scaling for logarithmic change of variables
                                           step*self.g.param*(grad_param_prior - grad_param_post), 
                                           *log_bounds)
                    self.update_param(torch.exp(self.eta))
                    
                    nit_param += 1 
                    if nit_mean > 0:  # update mean estimator
                        new_mean = (nit_mean*mean_hist[nit_mean-1] + g.param.item()) / (nit_mean + 1)
                        mean_hist[nit_mean] = new_mean
                        nit_mean += 1 

                        if abs(mean_hist[-1] - mean_hist[-2]) < tol:  # stop if mean estimator has converged
                            break

                    elif n >= it_burnin:
                        nit_mean = 1
                        mean_hist[0] = g.param.item()

        return (g.param.item(), param_hist[:nit_param].cpu(), mean_hist[:nit_mean].cpu(), 
                post_hist[:nit_param].cpu(), prior_hist[:nit_param].cpu())


### Forward model, noise

In [ ]:
# Set the global random seed from pytorch to ensure reproducibility of the example.
torch.manual_seed(0)

device = torch.device('cuda') #torch.device('cpu') #
print(device)
# Set up the variable to fetch dataset and operators.
dataset_name = "set3c"
img_size = 64
val_transform = transforms.Compose(
    [transforms.CenterCrop(img_size), transforms.ToTensor()]
)

dataset = load_dataset(dataset_name, transform=val_transform)

In [ ]:
im_ind = 0
x = dataset[im_ind][0].unsqueeze(0).to(device)

n_channels = 1
if n_channels == 1:
    deco = dinv.physics.Decolorize(device=device)
    x = deco(x)
dimx = x.shape[-1]*x.shape[-2]

filter_torch = dinv.physics.blur.gaussian_blur(sigma=(2, 2))
noise_level_img = 0.1  # Gaussian Noise standard deviation for the degradation

# The BlurFFT instance from physics enables to compute efficently backward operators with Fourier transform.
p = dinv.physics.Blur(
    img_size=(1, img_size, img_size),
    filter=filter_torch,
    device=device, padding="circular",
    noise_model=dinv.physics.GaussianNoise(sigma=noise_level_img))

# generate blurred, noisy measurement
y = p(x) 

class L2(Likelihood):
    def __init__(self, sigma):
        super().__init__()
        self.sigma = sigma
    
    def f(self, x, y):
        return 0.5 * torch.norm(y - p.A(x)) / 2 / self.sigma**2

    def grad(self, x, y):
        return p.A_adjoint(p.A(x) - y) / self.sigma**2

# plot the signal, the measurement and the noise
dinv.utils.plot([x, y, y - x], ["signal", "measurement", "diff"], rescale_mode='clip')

In [ ]:
# Lipschitz constant of nabla f
L_f = p.compute_norm(x0=torch.randn_like(x), tol=1e-5) / noise_level_img**2

# regularization parameter of proximity operator (lambda)
lam_reg = min(1/L_f, 2.)   


## Test 1 : wavelets

In [ ]:
class WaveletPrior(ParametrizedPrior):
    def __init__(self, param):
        super().__init__(param)
        self.dinv_tv = dinv.optim.prior.WaveletPrior(level=3, wv=["db{}".format(i) for i in range(8, 9)], p=1, device=device)

    def grad(self, x, lam_reg):
        return (x - self.dinv_tv.prox(x, self.param*lam_reg)) / lam_reg  

    def forward(self, x):
        return self.dinv_tv(x)*self.param

    def grad_param(self, x):
        return self.dinv_tv(x)


theta0 = torch.tensor(10.)

# Lipschitz constant of grad g
L_g =  1/lam_reg 

# Lipschitz constant of grad g + f
L =  L_f + L_g

# Stepsize of MCMC algorithm
gamma = 0.98*1/L
print("gamma: {}  lam: {}".format(gamma, lam_reg))
g = WaveletPrior(theta0)
f = L2(noise_level_img)

d0 = 1 / img_size**2 / theta0 
print("d0 = ", d0)
delta = lambda k: d0 / (k ** 0.4)

## Test 2 : RED

In [ ]:
class GSPnP(dinv.optim.prior.RED):
    r"""s
    Gradient-Step Denoiser prior.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.explicit_prior = True

    def forward(self, x, *args, **kwargs):
        r"""
        Computes the prior :math:`g(x)`.

        :param torch.tensor x: Variable :math:`x` at which the prior is computed.
        :return: (torch.tensor) prior :math:`g(x)`.
        """
        return self.denoiser.potential(x, *args, **kwargs)
path_ckpt = "GSDRUNet_grayscale_torch.ckpt" if n_channels == 1 else "GSDRUNet_torch.ckpt"
denoiser = dinv.models.GSDRUNet(pretrained=path_ckpt, 
                                in_channels=1, out_channels=1, device=device)
    
dinv_red = GSPnP(denoiser.to(device))

class RedPrior(ParametrizedPrior):
    def __init__(self, param, dinv_red_prior, sigma_denoiser=0.1):
        super().__init__(param)
        self.dinv_red_prior = dinv_red_prior
        self.sigma_denoiser = sigma_denoiser
        
    def grad(self, x, lam_reg):
        return self.param * self.dinv_red_prior.grad(x, sigma_denoiser=self.sigma_denoiser)

    def forward(self, x):
        return self.param * self.dinv_red_prior(x, sigma=self.sigma_denoiser) 

    def grad_param(self, x):
        return  self.dinv_red_prior(x, sigma=self.sigma_denoiser) 

In [ ]:
# SAPG parameters

# Lipschitz constant of nabla f
L_f = p.compute_norm(x0=torch.randn_like(x), tol=1e-5) / noise_level_img**2

# regularization parameter of proximity operator (lambda)
lam_reg = min(1/L_f, 2.)   

theta0 = torch.tensor(0.1)

# Lipschitz constant of grad g
L_g =  1

# Lipschitz constant of grad g + f
L =  L_f + L_g

# Stepsize of MCMC algorithm
gamma = 0.98*1/L
print("gamma: {}  lam: {}".format(gamma, lam_reg))
g = RedPrior(theta0, dinv_red)
f = L2(noise_level_img)

d0 = 1 / img_size**2 / theta0 
print("d0 = ", d0)
delta = lambda k: d0 / (k ** 0.4)

## Warming up the MC sampler

In [ ]:
g.update_param(theta0)
sapg = SAPG(y, g, f, gamma, gamma, lam_reg, X_init=p.A_A_adjoint(y).to(device).detach().clone())
nwarm = 15000
plot_stats = True
warm_prior = True
res = sapg.warm_up_post(nwarm, log_stats=plot_stats, thinning=1, burnin_ratio=0.95)
if warm_prior:
    res_prior = sapg.warm_up_prior(nwarm, log_stats=plot_stats, thinning=1, burnin_ratio=0.95)

if plot_stats:
    X_post, post = res
    plt.plot(post, label='post')
    if warm_prior:
        X_prior, prior = res_prior
        plt.plot(prior, label='prior')
    plt.legend();

In [ ]:
if plot_stats:
    print(X_post.shape)    
    start_ind, thinning = 0, 1
    print((len(X_post) - start_ind) // thinning)
    fig, ax = plt.subplots(3, 3, figsize=(13, 7))
    plot_MC_corr(X_post[start_ind::thinning, :].numpy(), ax=ax[0], title='Xpost')
    if warm_prior:
        plot_MC_corr(X_prior[start_ind::thinning, :].numpy(), ax=ax[1], title='Xprior')
        plot_MC_corr(prior[start_ind::thinning].numpy().reshape([-1, 1]), ax=ax[2], title='g(Xprior)')
    plt.tight_layout()

In [ ]:
if plot_stats:
    xmean = torch.mean(X_post.reshape([-1, img_size, img_size]), axis=0)
    dinv.utils.plot([xmean.reshape([1, img_size, img_size]), x, y])
    psnr = dinv.loss.metric.PSNR()
    print("PSNR : ", psnr(xmean.reshape([1, 1, img_size, img_size]), x.cpu()).item())

In [ ]:
param, param_hist, mean_hist, post_hist, prior_hist = sapg.run(delta, 1000, (0.0001, 100), init_param=theta0, 
                                                               thinning_post=20, thinning_prior=50, alpha=None, thinning_global=1, tol=1e-5)
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
ax[0].plot(-post_hist.numpy(), label=r'$g(X)$')
ax[0].plot(-prior_hist.numpy(), label=r'$g(\bar X)$')
ax[1].plot(param_hist.numpy(), label=r'$\theta$')
ax[2].plot(mean_hist.numpy(), label=r'$\bar{\theta}$')
ax[0].legend(), ax[1].legend(), ax[2].legend();
ax[0].set_yscale('log')
plt.tight_layout()

In [ ]:
X_hist, post_hist = sapg.sample_post(10000) 
print(X_hist[-1].shape)
mean_img = torch.tensor(torch.mean(X_hist[::10], axis=0), device=device)
dinv.utils.plot([mean_img, y, (x - mean_img)], ["reconstr", "measurement", "diff"], rescale_mode='clip')
psnr = dinv.loss.metric.PSNR()
print("PSNR : ", psnr(mean_img, x).item())